# Evaluation Pipeline

**Thesis:** RAG-Driven Natural Language Energy Demand Forecasting  
**Author:** Zoheb Anwar Hussain (Student ID: 1196931)

---

### What this notebook does
1. Computes **retrieval metrics** (Recall@k, Precision@k, MRR, nDCG)
2. Runs **hallucination keyword checks** (answer_must_include / answer_must_not_include)
3. Runs **RAGAS evaluation** (faithfulness, answer_relevancy, context_precision, context_recall)

### LLM Judge
RAGAS uses Llama 3.3 70B via Groq as the judge — independent of both the KB model
(Gemini 3 Flash) and the golden dataset model (Gemini 2.5 Flash).

---
## Cell 1 — Imports and Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from config import PATHS
from config.paths import create_all_directories
from src.utils import setup_logger, log_section
from src.embedding import get_embeddings_model
from src.rag import get_rag_llm
from src.evaluation import (
    compute_retrieval_metrics,
    check_hallucination,
    compute_hallucination_rate,
    run_ragas_evaluation,
)

logger = setup_logger("evaluation")
create_all_directories()
logger.info("All imports successful.")

2026-04-29 21:30:19 | INFO     | evaluation | All imports successful.


---
## Cell 2 — Load Data

In [2]:
log_section("Loading evaluation data", 1, 4)

# Golden dataset
golden_df = pd.read_csv(
    PATHS["golden_dataset"] / "combined_golden_dataset.csv"
)

# Retrieval results from Phase 4
retrieval_df = pd.read_csv(
    PATHS["retrieval_results"] / "retrieval_results.csv"
)

# RAG answers from Phase 5
rag_df = pd.read_csv(
    PATHS["rag_results"] / "rag_answers.csv"
)

logger.info(
    "Loaded: %d golden queries, %d retrieval results, %d RAG answers.",
    len(golden_df), len(retrieval_df), len(rag_df),
)

2026-04-29 21:30:45 | INFO     | evaluation | Loaded: 50 golden queries, 93 retrieval results, 93 RAG answers.



  STEP 1/4  [█░░░] 25%
  Loading evaluation data



---
## Cell 3 — Retrieval Metrics

Recall@k, Precision@k, MRR, nDCG — computed per query-pipeline combination.

In [3]:
log_section("Computing retrieval metrics", 2, 4)

retrieval_metrics_df = compute_retrieval_metrics(
    retrieval_results_df=retrieval_df,
    output_path=PATHS["evaluation_results"] / "retrieval_metrics.csv",
)


  STEP 2/4  [██░░] 50%
  Computing retrieval metrics


  RETRIEVAL METRICS BY PIPELINE
              recall_at_k  precision_at_k     mrr    ndcg
pipeline                                                 
dense              0.0603          0.0600  0.0907  0.0513
hierarchical       0.0808          0.0800  0.1592  0.0748
hybrid             0.0348          0.0348  0.0435  0.0323


---
## Cell 4 — Hallucination Checks

Keyword-based guards: answer_must_include and answer_must_not_include.

In [4]:
log_section("Running hallucination checks", 3, 4)

hallucination_df = compute_hallucination_rate(
    rag_results_df=rag_df,
    golden_df=golden_df,
    output_path=PATHS["evaluation_results"] / "hallucination_checks.csv",
)


  STEP 3/4  [███░] 75%
  Running hallucination checks


  HALLUCINATION CHECK BY PIPELINE
  dense           include_pass= 60.0%  exclude_pass= 98.0%  overall_pass= 58.0%
  hybrid          include_pass= 56.5%  exclude_pass= 95.7%  overall_pass= 52.2%
  hierarchical    include_pass= 65.0%  exclude_pass= 95.0%  overall_pass= 60.0%


---
## Cell 5 — RAGAS Evaluation

LLM-judged metrics: faithfulness, answer_relevancy, context_precision, context_recall.
Uses Llama 3.3 70B via Groq as the judge.

**Requires Groq API key.** Estimated runtime: ~10-15 minutes.

In [5]:
log_section("Running RAGAS evaluation", 4, 4)

# Initialise LLM judge and embeddings
llm        = get_rag_llm()
embeddings = get_embeddings_model()

ragas_df = run_ragas_evaluation(
    rag_results_df=rag_df,
    golden_df=golden_df,
    llm=llm,
    embeddings=embeddings,
    output_path=PATHS["evaluation_results"] / "ragas_scores.csv",
)

if ragas_df is not None:
    logger.info("RAGAS evaluation complete: %d scores.", len(ragas_df))
else:
    logger.warning(
        "RAGAS evaluation failed — check logs. "
        "Retrieval metrics and hallucination checks are still valid."
    )


  STEP 4/4  [████] 100%
  Running RAGAS evaluation



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/372 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[6]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kk66tp3pe01akvq9gny91wgz` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99916, Requested 1206. Please try again in 16m9.408s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[11]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kk66tp3pe01akvq9gny91wgz` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99885, Requested 1778. Please try again in 23m56.832s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception 


  RAGAS METRICS BY PIPELINE
              faithfulness  answer_relevancy  context_precision  context_recall
pipeline                                                                       
dense                  NaN            0.8219                NaN             NaN
hierarchical           NaN               NaN                NaN             NaN
hybrid                 NaN               NaN                NaN             NaN


---
## Pipeline Complete ✅

Evaluation results saved to:
```
outputs/evaluation_results/retrieval_metrics.csv      ← Recall@k, Precision@k, MRR, nDCG
outputs/evaluation_results/hallucination_checks.csv   ← Keyword inclusion/exclusion checks
outputs/evaluation_results/ragas_scores.csv           ← RAGAS faithfulness, relevancy, etc.
```

### Next Phase
Move to `notebooks/07_results_analysis.ipynb` to generate comparative
charts and tables for the thesis findings chapter.